# JUG vs PINT vs Tempo2: Noise Residuals Comparison

Side-by-side comparison of noise process realizations from **JUG**, **PINT**, and **Tempo2**
for **J0437-4715** (PPTA DR4, ~14.8k TOAs, red + DM + chromatic noise).

**Key convention difference:** JUG applies `TNDM_OFFSET ≈ 1.638` to convert
Tempo2's `TNDMAmp` to the enterprise amplitude convention. PINT does not.
We show PINT-raw, PINT-corrected (with TNDM_OFFSET), JUG, and Tempo2 for comparison.

**Note:** Tempo2 does not compute chromatic noise for this par file (`TNChromC` not set).

In [ ]:
import os
os.chdir('/home/mattm/soft/JUG')

import numpy as np
import matplotlib.pyplot as plt
import warnings, sys, math
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
%matplotlib inline

os.environ['JAX_PLATFORMS'] = 'cpu'
plt.rcParams.update({'figure.figsize': (14, 4), 'font.size': 11})

In [ ]:
par = 'data/pulsars/PPTA_data/ppta_dr4-data_dev-data-partim-MTM/data/partim/MTM/J0437-4715.par'
tim = 'data/pulsars/PPTA_data/ppta_dr4-data_dev-data-partim-MTM/data/partim/MTM/J0437-4715.tim'

# TNDM_OFFSET: Tempo2 DM basis uses 1/(DMKappa × freq_Hz²),
# enterprise/JUG uses (1400/freq_MHz)². This offset converts between them.
_DM_CONST = 2.41e-4
TNDM_OFFSET = math.log10(1400.0**2 * _DM_CONST / math.sqrt(12.0 * math.pi**2))
print(f'TNDM_OFFSET = {TNDM_OFFSET:.4f}')

## 1. Run PINT GLSFitter

In [ ]:
import pint.models, pint.toa, pint.logging, pint.fitter
pint.logging.setup(level='ERROR')

pint_model = pint.models.get_model(par, allow_T2=True, allow_tcb=True)
pint_toas = pint.toa.get_TOAs(tim, model=pint_model)
pint_model.find_empty_masks(pint_toas, freeze=True)
print(f'Loaded {pint_toas.ntoas} TOAs')

# --- PINT raw (no TNDM_OFFSET) ---
fitter_raw = pint.fitter.GLSFitter(pint_toas, pint_model)
fitter_raw.fit_toas(maxiter=1)
pint_noise_raw = fitter_raw.resids.noise_resids
pint_mjd = pint_toas.get_mjds().value

print('\nPINT (raw, no TNDM_OFFSET):')
for k, v in pint_noise_raw.items():
    print(f'  {k}: RMS = {np.std(v.value)*1e6:.4f} µs')
pint_resids_raw_us = fitter_raw.resids.calc_time_resids().to('s').value * 1e6
pint_total_raw_us = sum(v.value for v in pint_noise_raw.values()) * 1e6
print(f'  Noise-subtracted RMS: {np.std(pint_resids_raw_us - pint_total_raw_us):.4f} µs')

In [ ]:
# --- PINT corrected (with TNDM_OFFSET) ---
pint_model_corr = pint.models.get_model(par, allow_T2=True, allow_tcb=True)
pint_model_corr.find_empty_masks(pint_toas, freeze=True)
pint_model_corr.TNDMAMP.value = float(pint_model_corr.TNDMAMP.value) - TNDM_OFFSET
print(f'Corrected TNDMAmp: {pint_model_corr.TNDMAMP.value:.4f}')

fitter_corr = pint.fitter.GLSFitter(pint_toas, pint_model_corr)
fitter_corr.fit_toas(maxiter=1)
pint_noise_corr = fitter_corr.resids.noise_resids

print('\nPINT (corrected, with TNDM_OFFSET):')
for k, v in pint_noise_corr.items():
    print(f'  {k}: RMS = {np.std(v.value)*1e6:.4f} µs')
pint_resids_corr_us = fitter_corr.resids.calc_time_resids().to('s').value * 1e6
pint_total_corr_us = sum(v.value for v in pint_noise_corr.values()) * 1e6
print(f'  Noise-subtracted RMS: {np.std(pint_resids_corr_us - pint_total_corr_us):.4f} µs')

## 2. Run JUG GLS Fitter

In [ ]:
from jug.engine.session import TimingSession

session = TimingSession(par, tim)
jug_result = session.fit_parameters()

jug_nr = jug_result['noise_realizations']
jug_resids_us = np.array(jug_result['residuals_us'])
jug_mjd = np.array(jug_result['tdb_mjd'])

print('JUG noise realizations:')
for key in ['RedNoise', 'DMNoise', 'ChromaticNoise']:
    if key in jug_nr:
        print(f'  {key}: RMS = {np.std(np.array(jug_nr[key])):.4f} µs')

jug_total_us = sum(np.array(jug_nr[k]) for k in ['RedNoise', 'DMNoise', 'ChromaticNoise'] if k in jug_nr)
print(f'  Noise-subtracted RMS: {np.std(jug_resids_us - jug_total_us):.4f} µs')

## 2b. Run Tempo2

Run Tempo2 with the same par/tim files and extract noise realizations via `general2` plugin.
Tempo2 uses TCB natively (no conversion needed) and its own DM basis `1/(DMKappa × freq_Hz²)`.

In [ ]:
import subprocess, os

T2_BIN = '/home/mattm/miniforge3/envs/discotech/bin/tempo2'
T2_DATA = '/home/mattm/miniforge3/pkgs/tempo2-2025.02.1-hddb8a8a_0/share/tempo2'

env = os.environ.copy()
env['TEMPO2'] = T2_DATA

result = subprocess.run(
    [T2_BIN, '-f', par, tim,
     '-output', 'general2',
     '-s', '{sat} {freq} {post} {err} {tnrn} {tnrnerr} {tndm} {tndmerr} {tnchrom} {tnchromerr} {posttn}\n'],
    capture_output=True, text=True, env=env, timeout=300
)

# Parse output
t2_data = []
for line in result.stdout.strip().split('\n'):
    if line and line[0].isdigit():
        parts = line.split()
        if len(parts) >= 11:
            t2_data.append([float(x) for x in parts[:11]])
t2_data = np.array(t2_data)

t2_mjd      = t2_data[:, 0]
t2_freq     = t2_data[:, 1]
t2_post_s   = t2_data[:, 2]   # postfit residuals (seconds)
t2_red_s    = t2_data[:, 4]   # red noise realisation (seconds)
t2_dm_s     = t2_data[:, 6]   # DM noise realisation (seconds)
t2_chrom_s  = t2_data[:, 8]   # chromatic noise (seconds, all zero for this par)
t2_posttn_s = t2_data[:, 10]  # noise-subtracted residuals (seconds)

print(f'Tempo2: {len(t2_data)} TOAs')
print(f'Tempo2 noise realizations:')
print(f'  Red noise:  {np.std(t2_red_s)*1e6:.4f} µs')
print(f'  DM noise:   {np.std(t2_dm_s)*1e6:.4f} µs')
print(f'  Chrom noise:{np.std(t2_chrom_s)*1e6:.4f} µs (not computed, TNChromC missing)')
print(f'  Post-TN:    {np.std(t2_posttn_s)*1e6:.4f} µs')
print(f'  Postfit:    {np.std(t2_post_s)*1e6:.4f} µs')

## 3. Summary Table

In [ ]:
mapping = [
    ('Red Noise',   'RedNoise',        'pl_red_noise'),
    ('DM Noise',    'DMNoise',         'pl_DM_noise'),
    ('Chromatic',   'ChromaticNoise',  'pl_chrom_noise'),
]

print(f'{"Component":15s} {"JUG":>10s} {"PINT-raw":>10s} {"PINT-corr":>10s} {"Tempo2":>10s} {"JUG/Pcorr":>10s} {"JUG/T2":>10s}')
print(f'{"":15s} {"(µs)":>10s} {"(µs)":>10s} {"(µs)":>10s} {"(µs)":>10s} {"":>10s} {"":>10s}')
print('-' * 79)

t2_noise = {'RedNoise': t2_red_s, 'DMNoise': t2_dm_s, 'ChromaticNoise': t2_chrom_s}

for label, jkey, pkey in mapping:
    j = np.std(np.array(jug_nr[jkey]))
    pr = np.std(pint_noise_raw[pkey].value) * 1e6 if pkey in pint_noise_raw else 0
    pc = np.std(pint_noise_corr[pkey].value) * 1e6 if pkey in pint_noise_corr else 0
    t2 = np.std(t2_noise[jkey]) * 1e6
    r_pc = j / pc if pc > 0 else float('inf')
    r_t2 = j / t2 if t2 > 0 else float('inf')
    print(f'{label:15s} {j:10.4f} {pr:10.4f} {pc:10.4f} {t2:10.4f} {r_pc:10.4f} {r_t2:10.4f}')

jug_ns = np.std(jug_resids_us - jug_total_us)
pint_ns_raw = np.std(pint_resids_raw_us - pint_total_raw_us)
pint_ns_corr = np.std(pint_resids_corr_us - pint_total_corr_us)
t2_ns = np.std(t2_posttn_s) * 1e6
print('-' * 79)
print(f'{"Noise-sub RMS":15s} {jug_ns:10.4f} {pint_ns_raw:10.4f} {pint_ns_corr:10.4f} {t2_ns:10.4f}')

## 4. Time-Series Correlation

In [ ]:
print('=== JUG vs PINT-corrected ===')
for label, jkey, pkey in mapping:
    jts = np.array(jug_nr[jkey])
    pts = pint_noise_corr[pkey].value * 1e6 if pkey in pint_noise_corr else None
    if pts is not None and len(jts) == len(pts):
        corr = np.corrcoef(jts, pts)[0, 1]
        print(f'{label:15s}: r = {corr:.6f}, max|diff| = {np.max(np.abs(jts - pts)):.4f} µs')

print('\n=== JUG vs Tempo2 ===')
for label, jkey, pkey in mapping:
    jts = np.array(jug_nr[jkey])
    t2s = t2_noise[jkey] * 1e6
    if len(jts) == len(t2s) and np.std(t2s) > 0:
        corr = np.corrcoef(jts, t2s)[0, 1]
        print(f'{label:15s}: r = {corr:.6f}, max|diff| = {np.max(np.abs(jts - t2s)):.4f} µs')
    elif np.std(t2s) == 0:
        print(f'{label:15s}: Tempo2 = 0 (not computed)')

## 5. Noise Residuals vs MJD

Each row: one noise component. Columns: JUG, PINT-corrected, Tempo2.
Bottom row: noise-subtracted residuals.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 14), sharex=True)

components = [
    ('Red Noise',   'RedNoise',       'pl_red_noise'),
    ('DM Noise',    'DMNoise',        'pl_DM_noise'),
    ('Chromatic',   'ChromaticNoise', 'pl_chrom_noise'),
]

for i, (label, jkey, pkey) in enumerate(components):
    jts = np.array(jug_nr[jkey])
    pts_corr = pint_noise_corr[pkey].value * 1e6
    t2s = t2_noise[jkey] * 1e6
    ylim = max(np.max(np.abs(jts)), np.max(np.abs(pts_corr))) * 1.1
    if np.std(t2s) > 0:
        ylim = max(ylim, np.max(np.abs(t2s)) * 1.1)

    axes[i, 0].scatter(jug_mjd, jts, s=0.3, alpha=0.5, c='C0')
    axes[i, 0].set_ylabel(f'{label} (µs)')
    axes[i, 0].set_ylim(-ylim, ylim)
    if i == 0: axes[i, 0].set_title('JUG')

    axes[i, 1].scatter(pint_mjd, pts_corr, s=0.3, alpha=0.5, c='C1')
    axes[i, 1].set_ylim(-ylim, ylim)
    if i == 0: axes[i, 1].set_title('PINT (corrected DM)')

    axes[i, 2].scatter(t2_mjd, t2s, s=0.3, alpha=0.5, c='C2')
    axes[i, 2].set_ylim(-ylim, ylim)
    if i == 0: axes[i, 2].set_title('Tempo2')

# Bottom row: noise-subtracted
jug_ns_ts = jug_resids_us - jug_total_us
pint_ns_ts = pint_resids_corr_us - pint_total_corr_us
t2_ns_ts = t2_posttn_s * 1e6

axes[3, 0].scatter(jug_mjd, jug_ns_ts, s=0.3, alpha=0.5, c='C0')
axes[3, 0].set_ylabel('Noise-sub (µs)')
axes[3, 0].set_xlabel('MJD')
axes[3, 0].set_title(f'JUG (RMS={np.std(jug_ns_ts):.3f} µs)')

axes[3, 1].scatter(pint_mjd, pint_ns_ts, s=0.3, alpha=0.5, c='C1')
axes[3, 1].set_xlabel('MJD')
axes[3, 1].set_title(f'PINT-corr (RMS={np.std(pint_ns_ts):.3f} µs)')

axes[3, 2].scatter(t2_mjd, t2_ns_ts, s=0.3, alpha=0.5, c='C2')
axes[3, 2].set_xlabel('MJD')
axes[3, 2].set_title(f'Tempo2 (RMS={np.std(t2_ns_ts):.3f} µs)')

plt.tight_layout()
plt.savefig('notebooks/jug_vs_pint_vs_t2_noise.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Overlay: JUG vs PINT-corrected vs Tempo2

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

for i, (label, jkey, pkey) in enumerate(components):
    jts = np.array(jug_nr[jkey])
    pts = pint_noise_corr[pkey].value * 1e6
    t2s = t2_noise[jkey] * 1e6

    axes[i].scatter(jug_mjd, jts, s=0.5, alpha=0.4, c='C0', label=f'JUG ({np.std(jts):.2f} µs)')
    axes[i].scatter(pint_mjd, pts, s=0.5, alpha=0.4, c='C1', label=f'PINT-corr ({np.std(pts):.2f} µs)')
    if np.std(t2s) > 0:
        axes[i].scatter(t2_mjd, t2s, s=0.5, alpha=0.4, c='C2', label=f'Tempo2 ({np.std(t2s):.2f} µs)')
    axes[i].set_ylabel(f'{label} (µs)')
    axes[i].legend(loc='upper right', markerscale=5, fontsize=9)
    axes[i].set_title(label)

axes[-1].set_xlabel('MJD')
plt.tight_layout()
plt.savefig('notebooks/jug_vs_pint_vs_t2_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Differences: JUG−PINT-corr and JUG−Tempo2

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)

for i, (label, jkey, pkey) in enumerate(components):
    jts = np.array(jug_nr[jkey])
    pts = pint_noise_corr[pkey].value * 1e6
    t2s = t2_noise[jkey] * 1e6

    diff_pint = jts - pts
    axes[i, 0].scatter(jug_mjd, diff_pint, s=0.3, alpha=0.5, c='C3')
    axes[i, 0].axhline(0, color='k', lw=0.5, ls='--')
    axes[i, 0].set_ylabel(f'Δ{label} (µs)')
    axes[i, 0].set_title(f'JUG − PINT-corr: RMS={np.std(diff_pint):.4f} µs')

    if np.std(t2s) > 0:
        diff_t2 = jts - t2s
        axes[i, 1].scatter(jug_mjd, diff_t2, s=0.3, alpha=0.5, c='C4')
        axes[i, 1].axhline(0, color='k', lw=0.5, ls='--')
        axes[i, 1].set_title(f'JUG − Tempo2: RMS={np.std(diff_t2):.4f} µs')
    else:
        axes[i, 1].text(0.5, 0.5, 'Tempo2: not computed', transform=axes[i,1].transAxes, ha='center')
        axes[i, 1].set_title(f'JUG − Tempo2: N/A')

axes[-1, 0].set_xlabel('MJD')
axes[-1, 1].set_xlabel('MJD')
plt.tight_layout()
plt.savefig('notebooks/jug_vs_pint_vs_t2_diff.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Effect of TNDM_OFFSET: PINT-raw vs JUG

Without the TNDM_OFFSET correction, PINT's DM noise dominates everything.
This shows why the convention matters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, (label, jkey, pkey) in enumerate(components):
    jts = np.array(jug_nr[jkey])
    pts_raw = pint_noise_raw[pkey].value * 1e6
    pts_corr = pint_noise_corr[pkey].value * 1e6
    t2s = t2_noise[jkey] * 1e6

    axes[i].scatter(jug_mjd, jts, s=0.3, alpha=0.4, c='C0', label=f'JUG ({np.std(jts):.2f} µs)')
    axes[i].scatter(pint_mjd, pts_raw, s=0.3, alpha=0.3, c='C3', label=f'PINT-raw ({np.std(pts_raw):.2f} µs)')
    axes[i].scatter(pint_mjd, pts_corr, s=0.3, alpha=0.4, c='C1', label=f'PINT-corr ({np.std(pts_corr):.2f} µs)')
    if np.std(t2s) > 0:
        axes[i].scatter(t2_mjd, t2s, s=0.3, alpha=0.4, c='C2', label=f'Tempo2 ({np.std(t2s):.2f} µs)')
    axes[i].set_title(label)
    axes[i].set_xlabel('MJD')
    if i == 0:
        axes[i].set_ylabel('µs')
    axes[i].legend(loc='upper right', markerscale=5, fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/jug_vs_pint_vs_t2_tndm_offset_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Noise Smoothness Check

Sort by MJD and check sign-change fraction. Smooth noise should have few sign changes.
DM noise at a single reference frequency is checked (chromatic weighting causes apparent oscillation).

In [ ]:
def sign_change_frac(arr):
    """Fraction of consecutive sign changes in a sorted array."""
    signs = np.sign(arr)
    changes = np.sum(signs[1:] != signs[:-1])
    return changes / (len(arr) - 1)

sort_idx_jug = np.argsort(jug_mjd)
sort_idx_pint = np.argsort(pint_mjd)

print(f'{"Component":20s} {"JUG sign-changes":>16s} {"PINT-corr sign-changes":>22s}')
print('-' * 62)
for label, jkey, pkey in components:
    jts = np.array(jug_nr[jkey])[sort_idx_jug]
    pts = (pint_noise_corr[pkey].value * 1e6)[sort_idx_pint] if pkey in pint_noise_corr else None
    jsc = sign_change_frac(jts)
    psc = sign_change_frac(pts) if pts is not None else 0
    print(f'{label:20s} {jsc:15.1%} {psc:21.1%}')

print('\nNote: DM noise has high sign-change rate due to frequency mixing.')
print('Check achromatic (single-freq) sign changes for the DM component:')

# Get frequencies from session
jug_freqs = np.array([t.freq_mhz for t in session.toas_data])
freq_20cm = (jug_freqs > 1300) & (jug_freqs < 1500)
if np.sum(freq_20cm) > 100:
    sub_sort = np.argsort(jug_mjd[freq_20cm])
    jts_dm_20cm = np.array(jug_nr['DMNoise'])[freq_20cm][sub_sort]
    print(f'  DM noise (20cm band only): {sign_change_frac(jts_dm_20cm):.1%} sign changes ({np.sum(freq_20cm)} TOAs)')